In [2]:
!pip install -q osmnx geopandas shapely pyproj pandas numpy transformers datasets peft bitsandbytes accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.4/104.4 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.0 MB/s eta 0:00:00


In [3]:
import osmnx as ox
import pandas as pd

place_name = "Nancy, France"

tags = {
    'amenity': True,
    'tourism': True,
    'leisure': True
}

pois = ox.features_from_place(place_name, tags)

pois = pois[pois.geometry.type == 'Point'].copy()

pois['latitude'] = pois.geometry.y
pois['longitude'] = pois.geometry.x

df = pois[['name', 'latitude', 'longitude']].dropna()

df.to_csv("pois.csv", index=False)

In [4]:
import math

def haversine(lat1, lon1, lat2, lon2):
    R = 6371000
    phi1 = math.radians(lat1)
    phi2 = math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)

    a = math.sin(dphi/2)**2 + math.cos(phi1)*math.cos(phi2)*math.sin(dlambda/2)**2
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1-a))

    return R * c

In [5]:
def get_direction_4(lat1, lon1, lat2, lon2):
    dlat = lat2 - lat1
    dlon = lon2 - lon1

    if abs(dlat) > abs(dlon):
        if dlat > 0:
            return "north"
        else:
            return "south"
    else:
        if dlon > 0:
            return "east"
        else:
            return "west"

In [6]:
import pandas as pd

df = pd.read_csv("pois.csv")

relations = []

for i in range(len(df)):
    for j in range(i+1, len(df)):

        name1 = df.iloc[i]['name']
        name2 = df.iloc[j]['name']

        lat1 = df.iloc[i]['latitude']
        lon1 = df.iloc[i]['longitude']

        lat2 = df.iloc[j]['latitude']
        lon2 = df.iloc[j]['longitude']

        dist = haversine(lat1, lon1, lat2, lon2)

        if 50 < dist < 3000:  # good range
            direction = get_direction_4(lat1, lon1, lat2, lon2)

            relations.append({
                'poi_a': name1,
                'poi_b': name2,
                'distance_m': round(dist, 2),
                'direction': direction
            })

relations_df = pd.DataFrame(relations)

print("Total relations:", len(relations_df))

Total relations: 831693


In [7]:
relations_df = relations_df.sample(n=10000, random_state=42)

In [11]:
import numpy as np

unique_pois = df['name'].unique()
np.random.shuffle(unique_pois)

train_pois = unique_pois[:int(0.8 * len(unique_pois))]
val_pois = unique_pois[int(0.8 * len(unique_pois)):int(0.9 * len(unique_pois))]
test_pois = unique_pois[int(0.9 * len(unique_pois)):]

### Generate Relations per Split
For each subset of nodes: generate relations ONLY inside that subset

This ensures clean evaluation

In [12]:
def generate_relations(df, poi_subset):
    sub_df = df[df['name'].isin(poi_subset)]

    relations = []

    for i in range(len(sub_df)):
        for j in range(i+1, len(sub_df)):

            lat1 = sub_df.iloc[i]['latitude']
            lon1 = sub_df.iloc[i]['longitude']

            lat2 = sub_df.iloc[j]['latitude']
            lon2 = sub_df.iloc[j]['longitude']

            dist = haversine(lat1, lon1, lat2, lon2)

            if 50 < dist < 3000:
                relations.append({
                    'poi_a': sub_df.iloc[i]['name'],
                    'poi_b': sub_df.iloc[j]['name'],
                    'distance_m': round(dist, 2),
                    'direction': get_direction_4(lat1, lon1, lat2, lon2)
                })

    return pd.DataFrame(relations)

In [13]:
train_df = generate_relations(df, train_pois)
val_df = generate_relations(df, val_pois)
test_df = generate_relations(df, test_pois)

In [16]:
train_df = train_df.sample(n=10000, random_state=42)
val_df = val_df.sample(n=500, random_state=42)
test_df = test_df.sample(n=500, random_state=42)

In [17]:
import json

def convert(df):
    data = []
    for _, row in df.iterrows():
        data.append({
            "input": f"Where is {row['poi_b']} relative to {row['poi_a']}?",
            "output": f"{row['poi_b']} is {row['direction']} of {row['poi_a']} and about {row['distance_m']} meters away."
        })
    return data

json.dump(convert(train_df), open("train.json","w"))
json.dump(convert(val_df), open("val.json","w"))
json.dump(convert(test_df), open("test.json","w"))

In [19]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from datasets import load_dataset
from peft import LoraConfig, get_peft_model
import torch

model_name = "Qwen/Qwen2.5-0.5B"

# tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

# qLoRA config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

# model
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

# LoRA
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj","v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

# dataset
dataset = load_dataset("json", data_files={
    "train": "train.json",
    "validation": "val.json"
})

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

In [20]:
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['input', 'output'],
        num_rows: 10000
    })
    validation: Dataset({
        features: ['input', 'output'],
        num_rows: 500
    })
})


In [21]:
print(dataset["train"][0])

{'input': 'Where is Ambassy relative to City Burger?', 'output': 'Ambassy is north of City Burger and about 761.43 meters away.'}


In [22]:
def format_chat(example):
    return {
        "text": f"<|user|> {example['input']} <|assistant|> {example['output']}"
    }

dataset = dataset.map(format_chat)

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

In [26]:
def tokenize(example):
    text = f"<|user|> {example['input']} <|assistant|> {example['output']}"

    tokens = tokenizer(
        text,
        truncation=True,
        padding="max_length",
        max_length=128
    )

    labels = tokens["input_ids"].copy()


    user_text = f"<|user|> {example['input']} <|assistant|>"
    user_tokens = tokenizer(user_text, truncation=True, max_length=128)["input_ids"]

    labels[:len(user_tokens)] = [-100] * len(user_tokens)

    tokens["labels"] = labels

    return tokens

dataset = dataset.map(tokenize)

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

In [27]:
training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    learning_rate=2e-4,
    eval_strategy="steps",
    eval_steps=100
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"]
)

trainer.train()

Step,Training Loss,Validation Loss
100,No log,0.134447
200,No log,0.133702
300,No log,0.133333
400,No log,0.133574
500,0.378196,0.133233
600,0.378196,0.133384
700,0.378196,0.133183
800,0.378196,0.133134
900,0.378196,0.132882
1000,0.132858,0.132717


TrainOutput(global_step=1875, training_loss=0.1978819620768229, metrics={'train_runtime': 2966.7296, 'train_samples_per_second': 10.112, 'train_steps_per_second': 0.632, 'total_flos': 8258429583360000.0, 'train_loss': 0.1978819620768229, 'epoch': 3.0})

In [36]:
def generate(text):
    prompt = f"<|user|> {text} <|assistant|>"

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens=50,
        do_sample=False
    )

    result = tokenizer.decode(outputs[0], skip_special_tokens=True)

    return result.split("<|assistant|>")[-1].strip()

print(generate("Where is the Centre Commercial Saint Sébastien relative to the Marché Central?"))

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


The Centre Commercial Saint Sébastien is east of the Marché Central and about 1000.04 meters away.
